In [115]:
import json
with open('data_annotated.json', 'r', encoding='utf-8') as file:
    data = json.load(file)
with open('correct_predictions.json', 'r', encoding='utf-8') as file:
    data_true = json.load(file)    

In [116]:
def ratingToNumber(rating):
    if rating == "Strongly Agree":
        return 4
    elif rating == "Agree":
        return 3
    elif rating == "Disagree":
        return 2
    elif rating == "Strongly Disagree":
        return 1
    else:
        raise ValueError(f"Invalid rating: {rating}")


In [117]:
fn_labels = [
    "FN_misread_spec",
    "FN_added_requirement",
    "FN_overthink_edge",
    "FN_assumed_type",
    "FN_imagined_runtime",
    "FN_performance_nitpick",
    "FN_readability_nitpick",
    "FN_precision_error",
    "FN_boundary_error",
    "FN_logic_error",
    "FN_vague_description", 
    "FN_other",
]
count = {i: 0 for i in fn_labels}
q1_rating = {i: [] for i in fn_labels}
q2_rating = {i: [] for i in fn_labels}
q3_rating = {i: [] for i in fn_labels}
for i in data:
    for j in fn_labels:
        if i[j]:
            count[j] += 1
            q1_rating[j].append(ratingToNumber(i["q1_agreement"]))
            q2_rating[j].append(ratingToNumber(i["q2_agreement"]))
            q3_rating[j].append(ratingToNumber(i["q3_agreement"]))

count

{'FN_misread_spec': 64,
 'FN_added_requirement': 62,
 'FN_overthink_edge': 20,
 'FN_assumed_type': 0,
 'FN_imagined_runtime': 0,
 'FN_performance_nitpick': 0,
 'FN_readability_nitpick': 0,
 'FN_precision_error': 0,
 'FN_boundary_error': 0,
 'FN_logic_error': 31,
 'FN_vague_description': 5,
 'FN_other': 0}

In [118]:
fn_labels.append("True predictions")
q1_rating["True predictions"] = [ratingToNumber(i["q1_agreement"]) for i in data_true]
q2_rating["True predictions"] = [ratingToNumber(i["q2_agreement"]) for i in data_true]
q3_rating["True predictions"] = [ratingToNumber(i["q3_agreement"]) for i in data_true]
count["True predictions"] = len(data_true)

In [119]:
print(f"{'Label':22} {'size':>7} {'q1_avg':>7} {'q1_std':>7} {'q2_avg':>7} {'q2_std':>7} {'q3_avg':>7} {'q3_std':>7}")
print("-" * 65)
for j in fn_labels:
    if len(q1_rating[j]) > 0:
        import math
        import statistics

        def mean_std(lst):
            avg = statistics.mean(lst)
            std = statistics.stdev(lst)
            return avg, std

        q1_avg, q1_std = mean_std(q1_rating[j])
        q2_avg, q2_std = mean_std(q2_rating[j])
        q3_avg, q3_std = mean_std(q3_rating[j])
        print(f"{j:22} {count[j]:>7} {q1_avg:7.2f} {q1_std:7.2f} {q2_avg:7.2f} {q2_std:7.2f} {q3_avg:7.2f} {q3_std:7.2f}")
   
    else:
        print(f"{j:22}     ---    ---     ---     ---     ---     ---     ---")


Label                     size  q1_avg  q1_std  q2_avg  q2_std  q3_avg  q3_std
-----------------------------------------------------------------
FN_misread_spec             64    3.03    0.67    3.12    0.70    3.28    0.63
FN_added_requirement        62    3.13    0.64    3.29    0.61    3.27    0.63
FN_overthink_edge           20    3.20    0.62    3.30    0.57    3.30    0.57
FN_assumed_type            ---    ---     ---     ---     ---     ---     ---
FN_imagined_runtime        ---    ---     ---     ---     ---     ---     ---
FN_performance_nitpick     ---    ---     ---     ---     ---     ---     ---
FN_readability_nitpick     ---    ---     ---     ---     ---     ---     ---
FN_precision_error         ---    ---     ---     ---     ---     ---     ---
FN_boundary_error          ---    ---     ---     ---     ---     ---     ---
FN_logic_error              31    3.39    0.50    3.42    0.67    3.39    0.56
FN_vague_description         5    3.40    0.55    3.40    0.55    3.60 

In [120]:
from scipy.stats import ttest_ind
import numpy as np

trues = [1, 4, 2, 3, 5, 1, 3, 2, 6, 1, 1]
falses = [1, 4, 2, 3, 5, 8]

# Calculate the means
true_mean = np.mean(trues)
false_mean = np.mean(falses)

print("True mean:", true_mean)
print("False mean:", false_mean)

# Welch's independent-samples t-test
t_stat, p_value = ttest_ind(trues, falses, equal_var=False)

print("t-statistic:", t_stat)
print("p-value:", p_value)
def calculate_ttest(trues, falses):
    true_mean = np.mean(trues)
    false_mean = np.mean(falses)
    t_stat, p_value = ttest_ind(trues, falses, equal_var=False)
    return t_stat, p_value
calculate_ttest(trues, falses)


True mean: 2.6363636363636362
False mean: 3.8333333333333335
t-statistic: -1.0476126910446781
p-value: 0.3262581027582884


(np.float64(-1.0476126910446781), np.float64(0.3262581027582884))

In [125]:
print(f"{'Label':22} {'q1_ttest':>9} {'q1_pvalue':>9} {'q2_ttest':>9} {'q2_pvalue':>9} {'q3_ttest':>9} {'q3_pvalue':>9}")
for fn in fn_labels:
    if fn == "True predictions":
        continue
    q1_ttest, q1_pvalue = calculate_ttest(q1_rating[fn], q1_rating["True predictions"])
    q2_ttest, q2_pvalue = calculate_ttest(q2_rating[fn], q2_rating["True predictions"])
    q3_ttest, q3_pvalue = calculate_ttest(q3_rating[fn], q3_rating["True predictions"])
    print(f"{fn:22} {q1_ttest:7.2f} {q1_pvalue:7.2f} {q2_ttest:7.2f} {q2_pvalue:7.2f} {q3_ttest:7.2f} {q3_pvalue:7.2f}")


Label                   q1_ttest q1_pvalue  q2_ttest q2_pvalue  q3_ttest q3_pvalue
FN_misread_spec          -3.76    0.00   -2.51    0.01   -1.11    0.27
FN_added_requirement     -2.81    0.01   -0.94    0.35   -1.16    0.25
FN_overthink_edge        -1.37    0.18   -0.56    0.58   -0.61    0.55
FN_assumed_type            nan     nan     nan     nan     nan     nan
FN_imagined_runtime        nan     nan     nan     nan     nan     nan
FN_performance_nitpick     nan     nan     nan     nan     nan     nan
FN_readability_nitpick     nan     nan     nan     nan     nan     nan
FN_precision_error         nan     nan     nan     nan     nan     nan
FN_boundary_error          nan     nan     nan     nan     nan     nan
FN_logic_error           -0.15    0.88    0.33    0.74    0.04    0.97
FN_vague_description     -0.01    0.99    0.09    0.93    0.87    0.43
FN_other                   nan     nan     nan     nan     nan     nan


/opt/miniconda3/envs/dialogue/lib/python3.10/site-packages/numpy/_core/fromnumeric.py:3860: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/opt/miniconda3/envs/dialogue/lib/python3.10/site-packages/numpy/_core/_methods.py:145: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
/opt/miniconda3/envs/dialogue/lib/python3.10/site-packages/scipy/_lib/deprecation.py:234: SmallSampleWarning: One or more sample arguments is too small; all returned values will be NaN. See documentation for sample size requirements.
  return f(*args, **kwargs)


In [128]:
q1_all_fns = []
q2_all_fns = []
q3_all_fns = []
for fn in fn_labels:
    if fn == "True predictions":
        continue
    q1_all_fns.extend(q1_rating[fn])
    q2_all_fns.extend(q2_rating[fn])
    q3_all_fns.extend(q3_rating[fn])

q1_ttest, q1_pvalue = calculate_ttest(q1_all_fns, q1_rating["True predictions"])
q2_ttest, q2_pvalue = calculate_ttest(q2_all_fns, q2_rating["True predictions"])
q3_ttest, q3_pvalue = calculate_ttest(q3_all_fns, q3_rating["True predictions"])
print(f"{'All FN':22} {q1_ttest:7.2f} {q1_pvalue:7.2f} {q2_ttest:7.2f} {q2_pvalue:7.2f} {q3_ttest:7.2f} {q3_pvalue:7.2f}")

print(f"q1 : {mean_std(q1_all_fns)} q2 : {mean_std(q2_all_fns)} q3 : {mean_std(q3_all_fns)}")


All FN                   -3.51    0.00   -1.72    0.09   -1.15    0.25
q1 : (3.1538461538461537, 0.629356892190398) q2 : (3.258241758241758, 0.6516559330215566) q3 : (3.3076923076923075, 0.6073637935624387)


In [123]:
from scipy import stats
import statistics

TRUE_KEY = "True predictions"   # change to whatever key holds the correct responses

print(f"{'Q':>3} {'wrong_n':>8} {'wrong_avg':>10} {'right_n':>8} {'right_avg':>10} {'U':>9} {'p':>8} {'delta':>7}")
print("-" * 70)

for name, rat in [("q1", q1_rating), ("q2", q2_rating), ("q3", q3_rating)]:
    wrong = [x for j in fn_labels if j != TRUE_KEY for x in rat[j]]
    right = rat[TRUE_KEY]

    u, p = stats.mannwhitneyu(wrong, right, alternative="two-sided")

    # Cliff's delta from U: 0 = identical, +/-1 = no overlap
    delta = 2 * u / (len(wrong) * len(right)) - 1

    print(f"{name:>3} {len(wrong):>8} {statistics.mean(wrong):>10.2f} "
          f"{len(right):>8} {statistics.mean(right):>10.2f} "
          f"{u:>9.1f} {p:>8.3f} {delta:>+7.2f}")

  Q  wrong_n  wrong_avg  right_n  right_avg         U        p   delta
----------------------------------------------------------------------
 q1      182       3.15      154       3.40   11062.0    0.000   -0.21
 q2      182       3.26      154       3.38   12742.0    0.104   -0.09
 q3      182       3.31      154       3.38   13136.0    0.262   -0.06
